In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
df = pd.read_csv('salary_survey_raw.csv')

In [ ]:
# 1. Hàm phát hiện bằng IQR
def detect_outliers_iqr(df, col):
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = df[(df[col] < lower) | (df[col] > upper)]
    return outliers, lower, upper

# 2. Hàm phát hiện bằng Z-score
def detect_outliers_zscore(df, col, thresh=3):
    mean = df[col].mean()
    std = df[col].std()
    z_scores = (df[col] - mean) / std
    outliers = df[abs(z_scores) > thresh]
    return outliers

# Dữ liệu của chúng ta có các cột:
# 'how_old_are_you', 'annual_salary', etc.
# Cột annual_salary đang ở dạng chuỗi có dấu phẩy, nên cần chuyển sang numeric.

# Làm sạch dữ liệu trước:
# Loại bỏ dấu phẩy và chuyển sang dạng số
if 'annual_salary' in df.columns:
    df['annual_salary_num'] = df['annual_salary'].astype(str).str.replace(',', '').str.replace(' ', '').str.replace('$', '', regex=False)
    # Convert to numeric, setting errors to NaN
    df['annual_salary_num'] = pd.to_numeric(df['annual_salary_num'], errors='coerce')

# Chạy thử nghiệm và so sánh kết quả trên cột annual_salary_num
col = 'annual_salary_num'
if col in df.columns:
    print(f"=== PHÂN TÍCH CỘT: {col} ===")
    
    # Drop NaNs just for the outlier calculation to avoid issues
    df_clean = df.dropna(subset=[col])
    
    out_iqr, lower, upper = detect_outliers_iqr(df_clean, col)
    out_z = detect_outliers_zscore(df_clean, col, thresh=2) 
    
    print(f"IQR phát hiện ({len(out_iqr)} dòng): \n{out_iqr[col].values[:10]}...") # Print first 10 for brevity
    print(f"Z-score phát hiện ({len(out_z)} dòng): \n{out_z[col].values[:10]}...\n")

    # 3. Vẽ Boxplot minh họa
    plt.figure(figsize=(6, 4))
    sns.boxplot(data=df_clean, y=col, color='salmon')
    plt.title(f'Boxplot của {col}')
    plt.tight_layout()
    plt.show()
else:
    print("Không tìm thấy cột 'annual_salary_num'")

In [ ]:
# 4. Xử lý ngoại lệ (Outlier Treatment)

# Cách 1: Loại bỏ ngoại lệ (Trimming) dựa trên IQR
df_trimmed = df_clean[(df_clean[col] >= lower) & (df_clean[col] <= upper)].copy()
print(f"Số dòng ban đầu: {len(df_clean)}")
print(f"Số dòng sau khi xóa ngoại lệ (Trimming): {len(df_trimmed)}")

# Cách 2: Giới hạn giá trị biên (Capping / Winsorizing)
df_capped = df_clean.copy()
df_capped.loc[df_capped[col] < lower, col] = lower
df_capped.loc[df_capped[col] > upper, col] = upper
print(f"Số dòng sau khi Capping (giữ nguyên): {len(df_capped)}")

# Vẽ biểu đồ so sánh trước và sau khi xử lý
plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
sns.boxplot(data=df_clean, y=col, color='salmon')
plt.title('1. Trước khi xử lý (Original)')

plt.subplot(1, 3, 2)
sns.boxplot(data=df_trimmed, y=col, color='lightgreen')
plt.title('2. Sau khi xóa (Trimmed)')

plt.subplot(1, 3, 3)
sns.boxplot(data=df_capped, y=col, color='lightblue')
plt.title('3. Sau khi Capping')

plt.tight_layout()
plt.show()